# 03 — DistilBERT Fine-Tune Sweep

**Goal:** Fine-tune DistilBERT on Banking77 at 7 training-set sizes × 5 random seeds (35 runs total). Save per-(n, seed) test predictions so notebook 04 can compute the crossover with the Sonnet 4.6 baseline (macro-F1 = 0.8913, from notebook 02).

## Designed to run on either Colab or local

Cell 1 detects the environment. On Colab it clones the repo and installs missing packages. Locally it's a no-op (assumes the venv at `.venv/` already has the dependencies).

## Per-(n, seed) prediction files

Each of the 35 runs saves its test predictions to `results/finetune/n{n}_seed{seed}.parquet`. Benefits:
- **Resumable** — if Colab disconnects mid-sweep, skip-if-exists logic resumes from where it stopped
- **Atomic** — one disconnect costs at most one run
- **Mirrors the eval_responses/ pattern** from notebook 02 — consistent project structure

## Sections

1. Environment setup (Colab clone or local no-op)
2. Imports + device detection
3. Load static data: val, test, label names. Pre-tokenize once.
4. Hyperparameters + train_one_run helper
5. Main sweep loop — 35 runs, skip if predictions already saved
6. Aggregate per-(n, seed) parquets into `results/finetune_predictions.parquet`
7. Per-(n, seed) macro-F1 summary table
8. Save the best model checkpoint (n=5000, seed=0) for the Gradio demo + HF Hub upload
9. (Colab only) Zip + download results

## 1. Environment setup

Detects Colab. If on Colab: clones the repo (PUBLIC — requires no auth), installs the few packages Colab doesn't ship with, cds into the repo. Locally: assumes you have the venv at `.venv/` with requirements already installed.

In [ ]:
import os
import sys

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    REPO_URL = 'https://github.com/louisgrochla/ds-llm-judge-vs-small-model.git'
    REPO_DIR = '/content/ds-llm-judge-vs-small-model'
    if not os.path.exists(REPO_DIR):
        get_ipython().system(f'git clone {REPO_URL} {REPO_DIR}')
    os.chdir(REPO_DIR)
    get_ipython().system("pip install -q 'datasets>=2.16,<4.0' transformers accelerate scikit-learn")
    print(f'Colab — working in {REPO_DIR}')
else:
    print('Local — using existing venv at .venv/')

print(f'Working dir: {os.getcwd()}')

## 2. Imports + device detection

In [ ]:
import json
import math
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, Dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    get_linear_schedule_with_warmup,
    set_seed,
)
from sklearn.metrics import f1_score

sys.path.insert(0, '.')
from src.data import (
    load_test_set,
    load_val_set,
    load_train_subset,
    load_label_names,
    TRAINING_SIZES,
    N_SEEDS,
)

if torch.cuda.is_available():
    device = 'cuda'
elif torch.backends.mps.is_available():
    device = 'mps'
else:
    device = 'cpu'

print(f'Device:           {device}')
print(f'Training sizes:   {TRAINING_SIZES}')
print(f'Seeds:            {N_SEEDS}')
print(f'Total runs:       {len(TRAINING_SIZES) * N_SEEDS}')

FINETUNE_DIR = Path('results/finetune')
FINETUNE_DIR.mkdir(parents=True, exist_ok=True)

CKPT_PATH = Path('results/checkpoints/distilbert_banking77_n5000_seed0')

## 3. Load static data + pre-tokenize

Val and test are constant across the 35 runs — tokenize them once, reuse for every run. Saves significant CPU time at sweep scale.

In [ ]:
MODEL_NAME = 'distilbert-base-uncased'
MAX_LENGTH = 64
NUM_LABELS = 77

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

val_df = load_val_set()
test_df = load_test_set()
label_names = load_label_names()


def tokenize(df):
    enc = tokenizer(
        df['text'].tolist(),
        truncation=True,
        padding='max_length',
        max_length=MAX_LENGTH,
        return_tensors='pt',
    )
    return {
        'input_ids': enc['input_ids'],
        'attention_mask': enc['attention_mask'],
        'labels': torch.tensor(df['label'].astype(int).tolist(), dtype=torch.long),
    }


class TokenizedDataset(Dataset):
    def __init__(self, encoded):
        self.encoded = encoded

    def __len__(self):
        return len(self.encoded['labels'])

    def __getitem__(self, idx):
        return {k: v[idx] for k, v in self.encoded.items()}


val_encoded = tokenize(val_df)
test_encoded = tokenize(test_df)

print(f'Val:  {len(val_df):,} rows, {val_df["label"].nunique()} classes, tokenized')
print(f'Test: {len(test_df):,} rows, {test_df["label"].nunique()} classes, tokenized')

## 4. Hyperparameters + `train_one_run`

Fine-tune DistilBERT on one (n, seed). Returns metadata and writes predictions to `results/finetune/n{n}_seed{seed}.parquet`. If `save_model_path` is provided, also saves the model+tokenizer for later HF Hub upload.

Standard transformers training loop: AdamW + linear warmup, early stopping on val loss, best-checkpoint kept in memory.

In [ ]:
EPOCHS = 5
BATCH_SIZE = 32
LR = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_FRAC = 0.1
EARLY_STOPPING_PATIENCE = 2


def train_one_run(n: int, seed: int, save_model_path: Path | None = None) -> dict:
    set_seed(seed)

    train_df = load_train_subset(n, seed)
    train_encoded = tokenize(train_df)

    train_loader = DataLoader(TokenizedDataset(train_encoded), batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(TokenizedDataset(val_encoded), batch_size=BATCH_SIZE, shuffle=False)
    test_loader = DataLoader(TokenizedDataset(test_encoded), batch_size=BATCH_SIZE, shuffle=False)

    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=NUM_LABELS).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    total_steps = max(1, len(train_loader) * EPOCHS)
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(WARMUP_FRAC * total_steps),
        num_training_steps=total_steps,
    )

    best_val_loss = float('inf')
    best_state = None
    patience_left = EARLY_STOPPING_PATIENCE
    epochs_run = 0

    start_time = time.time()

    for epoch in range(EPOCHS):
        epochs_run += 1
        model.train()
        for batch in train_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            optimizer.zero_grad()
            out = model(**batch)
            out.loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()

        model.eval()
        val_losses = []
        with torch.no_grad():
            for batch in val_loader:
                batch = {k: v.to(device) for k, v in batch.items()}
                val_losses.append(model(**batch).loss.item())
        val_loss = float(np.mean(val_losses))

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.clone().detach().cpu() for k, v in model.state_dict().items()}
            patience_left = EARLY_STOPPING_PATIENCE
        else:
            patience_left -= 1
            if patience_left <= 0:
                break

    model.load_state_dict({k: v.to(device) for k, v in best_state.items()})

    model.eval()
    all_preds = []
    with torch.no_grad():
        for batch in test_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            all_preds.extend(model(**batch).logits.argmax(dim=-1).cpu().tolist())

    train_time_s = time.time() - start_time

    preds_df = test_df[['text', 'label']].copy()
    preds_df = preds_df.rename(columns={'label': 'true_label'})
    preds_df['pred_label'] = all_preds
    preds_df['true_intent'] = preds_df['true_label'].map(lambda i: label_names[int(i)])
    preds_df['pred_intent'] = preds_df['pred_label'].map(lambda i: label_names[int(i)])
    preds_df['is_correct'] = preds_df['true_label'] == preds_df['pred_label']
    preds_df['training_size'] = n
    preds_df['seed'] = seed

    out_path = FINETUNE_DIR / f'n{n}_seed{seed}.parquet'
    preds_df.to_parquet(out_path)

    metadata = {
        'training_size': n,
        'seed': seed,
        'epochs_run': epochs_run,
        'train_time_s': round(train_time_s, 1),
        'best_val_loss': round(best_val_loss, 4),
        'test_accuracy': float(preds_df['is_correct'].mean()),
        'test_macro_f1': float(f1_score(preds_df['true_label'], preds_df['pred_label'], average='macro', zero_division=0)),
    }

    if save_model_path is not None:
        save_model_path.mkdir(parents=True, exist_ok=True)
        model.save_pretrained(save_model_path)
        tokenizer.save_pretrained(save_model_path)

    del model, optimizer, scheduler
    if device == 'cuda':
        torch.cuda.empty_cache()

    return metadata


print(f'Hyperparameters: epochs={EPOCHS}, batch={BATCH_SIZE}, lr={LR}, warmup={WARMUP_FRAC}, patience={EARLY_STOPPING_PATIENCE}')

## 5. Main sweep loop

Iterates over (n, seed) pairs. Skips any combination whose prediction file already exists, so a Colab disconnect just means re-running this cell — completed runs aren't redone.

Saves the n=5000, seed=0 model to `results/checkpoints/distilbert_banking77_n5000_seed0/` for HF Hub upload. If that run's prediction file already exists but the checkpoint doesn't (e.g., you killed the kernel between the predictions save and the checkpoint save), the loop will re-train just that one run to capture the checkpoint.

In [ ]:
MAX_N = max(TRAINING_SIZES)
all_metadata = []
completed = 0
skipped = 0

for n in TRAINING_SIZES:
    for seed in range(N_SEEDS):
        preds_path = FINETUNE_DIR / f'n{n}_seed{seed}.parquet'
        is_checkpoint_run = (n == MAX_N and seed == 0)
        save_path = CKPT_PATH if is_checkpoint_run else None

        if preds_path.exists():
            if is_checkpoint_run and not CKPT_PATH.exists():
                print(f'  n={n:<5} seed={seed}: predictions exist but no model checkpoint — retraining to save model')
                meta = train_one_run(n, seed, save_model_path=save_path)
                completed += 1
                print(f'    done in {meta["train_time_s"]:.1f}s — test macro-F1 = {meta["test_macro_f1"]:.4f}')
                all_metadata.append(meta)
            else:
                skipped += 1
                print(f'  n={n:<5} seed={seed}: already done, skipping')
            continue

        print(f'  n={n:<5} seed={seed}: training...', end='', flush=True)
        meta = train_one_run(n, seed, save_model_path=save_path)
        completed += 1
        print(f' done in {meta["train_time_s"]:.1f}s — test macro-F1 = {meta["test_macro_f1"]:.4f}')
        all_metadata.append(meta)

print(f'\n{completed} runs completed this session, {skipped} skipped (already had predictions).')

if all_metadata:
    metadata_df = pd.DataFrame(all_metadata)
    metadata_path = FINETUNE_DIR / 'run_metadata.parquet'
    if metadata_path.exists():
        existing = pd.read_parquet(metadata_path)
        metadata_df = pd.concat([existing, metadata_df], ignore_index=True).drop_duplicates(subset=['training_size', 'seed'], keep='last')
    metadata_df.to_parquet(metadata_path)
    print(f'Saved {metadata_path}')

## 6. Aggregate per-(n, seed) parquets

Concatenates all 35 prediction files into one `results/finetune_predictions.parquet`. Notebook 04 joins this against the LLM predictions on `text` for the paired bootstrap test.

In [ ]:
all_files = sorted(FINETUNE_DIR.glob('n*_seed*.parquet'))
if not all_files:
    print('No prediction files yet — run section 5 first.')
else:
    dfs = [pd.read_parquet(p) for p in all_files]
    combined = pd.concat(dfs, ignore_index=True)
    combined_path = Path('results/finetune_predictions.parquet')
    combined.to_parquet(combined_path)
    expected = len(test_df)
    runs = len(all_files)
    print(f'Aggregated {runs} runs × {expected:,} test predictions = {len(combined):,} rows')
    print(f'Saved {combined_path}')

## 7. Per-(n, seed) macro-F1 summary

In [ ]:
if not all_files:
    print('No runs to summarise.')
else:
    summary_rows = []
    for p in all_files:
        df = pd.read_parquet(p)
        n = int(df['training_size'].iloc[0])
        seed = int(df['seed'].iloc[0])
        summary_rows.append({
            'training_size': n,
            'seed': seed,
            'accuracy': float(df['is_correct'].mean()),
            'macro_f1': float(f1_score(df['true_label'], df['pred_label'], average='macro', zero_division=0)),
        })
    summary = pd.DataFrame(summary_rows)

    print('macro-F1 per (n, seed):')
    print(summary.pivot(index='training_size', columns='seed', values='macro_f1').round(3).to_string())
    print()
    print('Mean ± std across seeds at each n:')
    agg = summary.groupby('training_size')['macro_f1'].agg(['mean', 'std']).round(4)
    print(agg.to_string())
    print()
    print(f'Sonnet 4.6 baseline (from notebook 02): 0.8913 macro-F1 on test')

## 8. Verify the model checkpoint is saved

The n=5000, seed=0 model lands at `results/checkpoints/distilbert_banking77_n5000_seed0/`. Used in the Gradio demo and published to HF Hub as the final artefact.

In [ ]:
if CKPT_PATH.exists():
    files_in_ckpt = sorted(p.name for p in CKPT_PATH.iterdir())
    size_mb = sum(p.stat().st_size for p in CKPT_PATH.iterdir()) / (1024 * 1024)
    print(f'Checkpoint at {CKPT_PATH}/ — {size_mb:.1f} MB, files: {files_in_ckpt}')
else:
    print(f'No checkpoint at {CKPT_PATH}/ — run section 5 to train n=5000, seed=0 with save_model_path set')

## 9. (Colab only) Zip + download results

Bundles everything notebook 04 needs (per-(n, seed) prediction files, aggregated parquet, run metadata, model checkpoint) into a single zip. Triggers a browser download — unzip on your Mac, commit + push from there.

In [ ]:
if not IN_COLAB:
    print('Local — results already in your working directory; skip the download step.')
else:
    import shutil

    ZIP_BASE = '/content/finetune_results'
    if os.path.exists(ZIP_BASE):
        shutil.rmtree(ZIP_BASE)
    os.makedirs(f'{ZIP_BASE}/results/finetune', exist_ok=True)
    os.makedirs(f'{ZIP_BASE}/results/checkpoints', exist_ok=True)

    for p in Path('results/finetune').glob('*.parquet'):
        shutil.copy(p, f'{ZIP_BASE}/results/finetune/')
    if Path('results/finetune_predictions.parquet').exists():
        shutil.copy('results/finetune_predictions.parquet', f'{ZIP_BASE}/results/')
    if CKPT_PATH.exists():
        shutil.copytree(CKPT_PATH, f'{ZIP_BASE}/results/checkpoints/{CKPT_PATH.name}')

    shutil.make_archive('/content/finetune_results', 'zip', ZIP_BASE)
    from google.colab import files as colab_files
    colab_files.download('/content/finetune_results.zip')
    print('Downloaded finetune_results.zip — unzip into the repo root on your Mac, then git add results/ and push.')